## TEST OF 3-STEP-STRATEGY
The main objective of this notebook is to start testing the possible chunking alternatives
with various documents in data/raw_docs directory and see which one fits the best to their metadata assignment.
The steps proposed to do this in a possible final solution are:
    1.  Do a full document analysis to determine its metadata.
    2.  Apply the chunking to it.
    3.  Do a single-chunk metadata analysis to focus on a more well separated information.


## METADATA FIELDS

### FOR THE FILTER SEARCH
**clearance_level** = [0..3] <br>
    0: public (everybody) <br>
    1: intern (only employees, not customers)<br>
    2: confidential (only certain departments) <br>
    3: strict (only executives and/or document owners)

**allowed_departments** (list of which departments can see this is there's a >2 level)

### GLOBAL CONTEXT (Whole file)
**source_file**: name of the file <br>

**doc_type**: ("report", "manual", "contract"...) <br>

**global_topic**: ("hardware_specs", ...)

### LOCAL CONTEXT (Specific chunks)
**chunk_id**: unique identifier reference <br>

**page_number**: reference to where the chunk lives <br>

**contains_PII**: boolean → true if there's passwords, names, ids

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os

DOCS_DIR = "../../../data/raw_docs/"

### STEP 1

In [9]:
def global_tagging(filename, department, confidentiality, doc_type, doc_topic):
    file_path = os.path.join(DOCS_DIR, filename)
    #HERE SHOULD BE A DOCUMENT TYPE DETECTION (pdf; excel;...)
    #for this case let's assume only PDF format
    loader = PyPDFLoader(file_path)

    #divide the document by its pages
    pages = loader.load()

    for page in pages:
        #LangChain extracts by default the fields "source" and "page"
        #HERE Let's suppose for this test that the department;confidentiality;type;topic levels are given or visible/specified in the document previously so I don't have to build an agent detector as for now.
        match confidentiality:
            case "public":
                page.metadata["clearance_level"] = 0

            case "intern":
                page.metadata["clearance_level"] = 1

            case "confidential":
                page.metadata["clearance_level"] = 2

            case "strict":
                page.metadata["clearance_level"] = 3

            case _:
                print("Not a valid level of confidentiality")
                exit()

        if page.metadata["clearance_level"] >= 2:
            page.metadata["department"] = department
        else:
            page.metadata["department"] = "NaN"

        page.metadata["doc_type"] = doc_type
        page.metadata["doc_topic"] = doc_topic

    print(f"Document {filename} loaded with {len(pages)} pages")
    print(f"The metadata fields assigned in this page are: {pages[0].metadata}")

    return pages

#RUN TEST with the documents
try:
    tech_docs = global_tagging("Witty-QuickGuide-EN.pdf", "engineering", "public", "manual", "a Witty device guide")
    fin_docs = global_tagging("Witty-Financial-Report-2025.pdf", "finance", "strict", "report", "a Witty company financial report of 2025")
except Exception as e:
    print("No existe ese documento")




Document Witty-QuickGuide-EN.pdf loaded with 2
The metadata fields assigned in this page are: {'source': '../data/raw_docs/Witty-QuickGuide-EN.pdf', 'page': 0, 'clearance_level': 0, 'department': 'NaN', 'doc_type': 'manual', 'doc_topic': 'a Witty device guide'}
Document Witty-Financial-Report-2025.pdf loaded with 2
The metadata fields assigned in this page are: {'source': '../data/raw_docs/Witty-Financial-Report-2025.pdf', 'page': 0, 'clearance_level': 3, 'department': 'finance', 'doc_type': 'report', 'doc_topic': 'a Witty company financial report of 2025'}
